In [49]:
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
from tqdm import tqdm

In [50]:
train = pd.read_csv("dataset/train_data.csv")
test = pd.read_csv("dataset/test_data.csv")

print(train.head())
print(test.head())

     pair_id                                          py_source  \
0  pair_0000  def f_gold ( num , divisor ) :\n    while ( nu...   
1  pair_0001  def f_gold ( arr , n ) :\n    mp = dict ( )\n ...   
2  pair_0002  def f_gold ( s ) :\n    length = len ( s )\n  ...   
3  pair_0003  def f_gold ( arr , n ) :\n    arr.sort ( ) ;\n...   
4  pair_0004  import sys\n\ndef f_gold ( arr , n ) :\n    re...   

                                          cpp_source  
0  using namespace std;\nint f_gold ( int num, in...  
1  using namespace std;\nint f_gold ( int arr [ ]...  
2  using namespace std;\nint f_gold ( string str ...  
3  using namespace std;\nint f_gold ( int arr [ ]...  
4  using namespace std;\nint f_gold ( int arr [ ]...  
    type          datapointID  \
0  query  py_test_public_0000   
1  query  py_test_public_0001   
2  query  py_test_public_0002   
3  query  py_test_public_0003   
4  query  py_test_public_0004   

                                              source  
0  def f_gold

In [51]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

class TextDataset(Dataset):
    def __init__(self):
        super().__init__()

        self.py = tokenizer(
            train["py_source"].tolist(), 
            padding=True,
            truncation=True,
            return_tensors="pt"
            )
        
        self.cpp = tokenizer(
            train["cpp_source"].tolist(), 
            padding=True,
            truncation=True,
            return_tensors="pt"
            )
        
    def __len__(self):
        return len(self.py)
    
    def __getitem__(self, index):
        return self.py["input_ids"][index], self.py["attention_mask"][index], self.cpp["input_ids"][index], self.cpp["attention_mask"][index],

dataset = TextDataset()
loader = DataLoader(dataset, batch_size=8)

In [52]:
class CodeEmbedder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained("bert-base-uncased")

    def mean_pool(self, out, mask):
        token_emb = out.last_hidden_state
        mask = mask.unsqueeze(-1)
        return (token_emb * mask).sum(1) / mask.sum(1)

    def forward(self, input_ids, attention_mask, **kwargs):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        emb = self.mean_pool(out, attention_mask)
        emb = F.normalize(emb, dim=1)
        return emb

In [53]:
device = "mps"
model = CodeEmbedder().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

for epoch in range(50):
    total_loss = 0

    model.train()
    for py_ids, py_att, cpp_ids, cpp_att in loader:
        py_ids = py_ids.to(device)
        py_att = py_att.to(device)

        cpp_ids = cpp_ids.to(device)
        cpp_att = cpp_att.to(device)

        py_emb = model(py_ids, py_att)
        cpp_emb = model(cpp_ids, cpp_att)

        logits = py_emb @ cpp_emb.T
        labels = torch.arange(logits.size(0), device=device)

        loss1 = F.cross_entropy(logits, labels)
        loss2 = F.cross_entropy(logits.T, labels)

        loss = (loss1 + loss2) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(epoch, total_loss / len(loader))
        

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16611.27it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0 1.0420942306518555
1 0.9923561811447144
2 0.9280391931533813
3 0.8362274169921875
4 0.7566487789154053
5 0.6661399006843567
6 0.6056731939315796
7 0.5591471195220947
8 0.5192804932594299
9 0.48769694566726685
10 0.4623885452747345
11 0.4478222131729126
12 0.4429037570953369
13 0.4277283549308777
14 0.41997963190078735
15 0.41679883003234863
16 0.41134005784988403
17 0.4046894907951355
18 0.4015139937400818
19 0.3963525891304016
20 0.3934308588504791
21 0.39080703258514404
22 0.38774555921554565
23 0.38675379753112793
24 0.385900616645813
25 0.38299643993377686
26 0.38125139474868774
27 0.38129812479019165
28 0.37966305017471313
29 0.37838226556777954
30 0.37824296951293945
31 0.37750470638275146
32 0.3761395812034607
33 0.3754745125770569
34 0.3748396039009094
35 0.3749599754810333
36 0.37419766187667847
37 0.3742038607597351
38 0.37463638186454773
39 0.37372952699661255
40 0.3738868832588196
41 0.3736213445663452
42 0.3734191656112671
43 0.37271785736083984
44 0.37293577194213867
45

In [54]:
queries = test[test["type"] == "query"].reset_index(drop=True)
candidates = test[test["type"] == "candidate"].reset_index(drop=True)

candidate_ids = candidates["datapointID"].tolist()
py_ids = queries["datapointID"].tolist()
query_sources = queries["source"].tolist()


class CppDataset(Dataset):
    def __init__(self):
        self.cpp = tokenizer(
            candidates["source"].tolist(),
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

    def __len__(self):
        return self.cpp["input_ids"].shape[0]

    def __getitem__(self, index):
        return {
            "input_ids": self.cpp["input_ids"][index],
            "attention_mask": self.cpp["attention_mask"][index]
        }


cpp_loader = DataLoader(CppDataset(), batch_size=8)

cpp_embs = []

model.eval()

with torch.no_grad():
    for batch in cpp_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        emb = model(**batch)
        cpp_embs.append(emb.cpu())

cpp_embs = torch.cat(cpp_embs, dim=0)


answers = []

model.eval()

with torch.no_grad():
    for query_id, query in tqdm(zip(py_ids, query_sources), total=len(query_sources)):
        enc = tokenizer(
            query,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        py_emb = model(**enc).cpu()

        scores = py_emb @ cpp_embs.T
        ranked_idx = scores.argsort(dim=1, descending=True)[0]

        ranked_candidate_ids = [
            candidate_ids[j]
            for j in ranked_idx.tolist()
        ]

        answers.append({
            "subtaskID": 1,
            "datapointID": query_id,
            "answer": ";".join(ranked_candidate_ids)
        })


submission = pd.DataFrame(answers)
submission.to_csv("submission.csv", index=False)

100%|██████████| 200/200 [00:08<00:00, 24.61it/s]
